In [0]:
from pyspark.sql import functions as F

df = spark.table("workspace.default.transactions_clean")

# Set a reference "analysis date" - one day after the last transaction in the dataset
max_date = df.select(F.max("invoice_date")).collect()[0][0]
analysis_date = max_date + F.expr("INTERVAL 1 DAY")

print(f"Dataset max date: {max_date}")

# Calculate RFM per customer
rfm = df.groupBy("customer_id").agg(
    F.datediff(F.lit(max_date), F.max("invoice_date")).alias("recency_days"),
    F.countDistinct("invoice").alias("frequency"),
    F.sum("line_total").alias("monetary")
)

print(f"Customers in RFM table: {rfm.count():,}")
rfm.show(10)

Dataset max date: 2011-12-09 12:50:00
Customers in RFM table: 5,878
+-----------+------------+---------+------------------+
|customer_id|recency_days|frequency|          monetary|
+-----------+------------+---------+------------------+
|      17804|         376|        3|            385.23|
|      17742|         113|       11|           2098.48|
|      16887|          35|        3|            660.72|
|      15380|           8|       11| 3479.060000000001|
|      12980|         157|       21|16245.779999999999|
|      16210|           1|       35|          38489.51|
|      13089|           2|      203|116737.86000000002|
|      12437|           1|       39|12683.400000000001|
|      13931|          95|       11| 5257.380000000002|
|      15194|           3|       28|          12427.69|
+-----------+------------+---------+------------------+
only showing top 10 rows


In [0]:
from pyspark.sql.window import Window

# Recency: LOWER days = BETTER, so we reverse the scoring (5 = most recent)
# Frequency & Monetary: HIGHER = BETTER (5 = most frequent/highest spend)

rfm_scored = rfm.withColumn(
    "r_score", F.ntile(5).over(Window.orderBy(F.desc("recency_days")))
).withColumn(
    "f_score", F.ntile(5).over(Window.orderBy("frequency"))
).withColumn(
    "m_score", F.ntile(5).over(Window.orderBy("monetary"))
)

rfm_scored = rfm_scored.withColumn(
    "rfm_segment_score",
    F.concat(F.col("r_score"), F.col("f_score"), F.col("m_score"))
)

rfm_scored.select("customer_id", "recency_days", "frequency", "monetary",
                   "r_score", "f_score", "m_score", "rfm_segment_score").show(10)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-----------+------------+---------+-----------------+-------+-------+-------+-----------------+
|customer_id|recency_days|frequency|         monetary|r_score|f_score|m_score|rfm_segment_score|
+-----------+------------+---------+-----------------+-------+-------+-------+-----------------+
|      14095|         722|        1|             2.95|      1|      1|      1|              111|
|      13788|         505|        1|             3.75|      1|      1|      1|              111|
|      16738|         297|        1|             3.75|      2|      1|      1|              211|
|      14792|          63|        1|              6.2|      3|      2|      1|              321|
|      15913|         534|        1|6.300000000000001|      1|      1|      1|              111|
|      15040|         541|        1|             7.49|      1|      1|      1|              111|
|      18115|         697|        1|              9.7|      1|      1|      1|              111|
|      17378|         374|    

In [0]:
rfm_final = rfm_scored.withColumn(
    "customer_segment",
    F.when((F.col("r_score") >= 4) & (F.col("f_score") >= 4) & (F.col("m_score") >= 4), "Champions")
     .when((F.col("r_score") >= 3) & (F.col("f_score") >= 3), "Loyal Customers")
     .when((F.col("r_score") >= 4) & (F.col("f_score") <= 2), "New Customers")
     .when((F.col("r_score") <= 2) & (F.col("f_score") >= 3), "At Risk")
     .when((F.col("r_score") <= 2) & (F.col("f_score") <= 2) & (F.col("m_score") <= 2), "Lost")
     .otherwise("Needs Attention")
)

# Check segment distribution
rfm_final.groupBy("customer_segment").agg(
    F.count("*").alias("customer_count"),
    F.round(F.avg("monetary"), 2).alias("avg_spend")
).orderBy(F.desc("customer_count")).show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+----------------+--------------+---------+
|customer_segment|customer_count|avg_spend|
+----------------+--------------+---------+
| Loyal Customers|          1473|  1959.04|
|            Lost|          1362|    263.8|
|       Champions|          1345|  9080.86|
|         At Risk|           708|  2209.02|
| Needs Attention|           701|   864.35|
|   New Customers|           289|   397.31|
+----------------+--------------+---------+



In [0]:
rfm_final.write.format("delta").mode("overwrite").saveAsTable("workspace.default.customer_rfm_segments")
print("✅ Saved customer_rfm_segments table")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


✅ Saved customer_rfm_segments table


In [0]:
# Product-level performance: revenue, units sold, order count, unique customers
product_performance = df.groupBy("stock_code", "description").agg(
    F.sum("line_total").alias("total_revenue"),
    F.sum("quantity").alias("total_units_sold"),
    F.countDistinct("invoice").alias("order_count"),
    F.countDistinct("customer_id").alias("unique_customers"),
    F.round(F.avg("price"), 2).alias("avg_price")
).orderBy(F.desc("total_revenue"))

print(f"Total unique products: {product_performance.count():,}")
product_performance.show(10, truncate=40)

Total unique products: 5,315
+----------+----------------------------------+------------------+----------------+-----------+----------------+---------+
|stock_code|                       description|     total_revenue|total_units_sold|order_count|unique_customers|avg_price|
+----------+----------------------------------+------------------+----------------+-----------+----------------+---------+
|     22423|          REGENCY CAKESTAND 3 TIER| 286486.3000000012|           24899|       3317|            1314|    12.46|
|    85123A|WHITE HANGING HEART T-LIGHT HOLDER|252072.46000000404|           93640|       4888|            1490|     2.87|
|     23843|       PAPER CRAFT , LITTLE BIRDIE|          168469.6|           80995|          1|               1|     2.08|
|         M|                            Manual|152340.56999999983|            9803|        620|             438|   206.44|
|    85099B|           JUMBO BAG RED RETROSPOT|136980.08000000124|           75759|       2612|             86

In [0]:
country_sales = df.groupBy("country").agg(
    F.sum("line_total").alias("total_revenue"),
    F.countDistinct("customer_id").alias("unique_customers"),
    F.countDistinct("invoice").alias("order_count")
).orderBy(F.desc("total_revenue"))

country_sales.show(20, truncate=False)

+---------------+--------------------+----------------+-----------+
|country        |total_revenue       |unique_customers|order_count|
+---------------+--------------------+----------------+-----------+
|United Kingdom |1.4723147517003272E7|5350            |33541      |
|EIRE           |621631.1100000123   |5               |567        |
|Netherlands    |554232.3400000028   |22              |228        |
|Germany        |431262.4610000001   |107             |789        |
|France         |355257.46999999945  |95              |614        |
|Australia      |169968.10999999972  |15              |95         |
|Spain          |109178.52999999968  |41              |154        |
|Switzerland    |100365.34000000003  |22              |90         |
|Sweden         |91549.71999999986   |19              |104        |
|Denmark        |69862.19000000008   |12              |43         |
|Belgium        |65753.42000000033   |29              |149        |
|Norway         |60109.620000000126  |13        

In [0]:
monthly_sales = df.withColumn(
    "year_month", F.date_format("invoice_date", "yyyy-MM")
).groupBy("year_month").agg(
    F.sum("line_total").alias("total_revenue"),
    F.countDistinct("invoice").alias("order_count"),
    F.countDistinct("customer_id").alias("unique_customers")
).orderBy("year_month")

monthly_sales.show(30)

+----------+------------------+-----------+----------------+
|year_month|     total_revenue|order_count|unique_customers|
+----------+------------------+-----------+----------------+
|   2009-12| 686654.1599999949|       1512|             955|
|   2010-01| 557319.0620000134|       1011|             720|
|   2010-02|506371.06600001536|       1104|             772|
|   2010-03| 699608.9909999889|       1524|            1057|
|   2010-04| 594609.1919999976|       1329|             942|
|   2010-05| 599985.7900000075|       1377|             966|
|   2010-06| 639066.5800000058|       1497|            1041|
|   2010-07| 591636.7400000024|       1381|             928|
|   2010-08| 604242.6499999989|       1293|             911|
|   2010-09| 831615.0009999905|       1689|            1145|
|   2010-10|1036679.9999999028|       2133|            1497|
|   2010-11|1172336.0419998672|       2587|            1607|
|   2010-12| 884591.8899999729|       1400|             885|
|   2011-01| 569445.0400

In [0]:
product_performance.write.format("delta").mode("overwrite").saveAsTable("workspace.default.product_performance")
country_sales.write.format("delta").mode("overwrite").saveAsTable("workspace.default.country_sales")
monthly_sales.write.format("delta").mode("overwrite").saveAsTable("workspace.default.monthly_sales")

print("✅ All Week 2 tables saved")

✅ All Week 2 tables saved
